# StudyMate – test the project on Google Colab
Project 1E – Course Study Assistant. Run the cells **in order** (Runtime → Run all also works, except cell 1 asks you to upload the ZIP).

**What this notebook checks:** 1) the data, 2) PDF extraction with pypdf, 3) training, 4) evaluation, 5) the automatic tests, 6) the search and classifier by hand, 7) the Flask website running inside Colab.

## 1. Upload the project ZIP
Choose `StudyMate_Project_1E.zip`. (Or put it in Google Drive and use the Drive option in the next cell.)

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick StudyMate_Project_1E.zip
ZIP = list(uploaded)[0]
print("uploaded:", ZIP)

In [ ]:
# OPTION B (instead of the upload): the ZIP is in Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# ZIP = '/content/drive/MyDrive/StudyMate_Project_1E.zip'

In [ ]:
!rm -rf /content/studymate && mkdir -p /content/studymate
!unzip -q -o "$ZIP" -d /content/studymate
%cd /content/studymate
!ls

## 2. Install the libraries

In [ ]:
!pip -q install -r requirements.txt

## 3. Check the data (project requirements)

In [ ]:
import pandas as pd
P = pd.read_csv('data/AI/passages.csv'); L = pd.read_csv('data/AI/questions_labeled.csv')
B = pd.read_csv('data/AI/question_bank.csv'); S = pd.read_csv('data/AI/search_test.csv'); O = pd.read_csv('data/AI/offtopic_test.csv')
print(f"passages: {len(P)} from {P.lecture.nunique()} lectures")
print(f"labelled questions: {len(L)} (need >= 300)  topics: {L.topic.nunique()}")
print(f"question bank: {len(B)}, min per topic: {B.groupby('topic').size().min()} (need >= 5 for >= 8 topics)")
print(f"search test: {len(S)} (need 30)   off-topic: {len(O)} (need 10)")
L.topic.value_counts()

## 4. Rebuild everything from the lecture PDFs
Extract slides → train → evaluate. This replaces the models that came in the ZIP, so you prove the whole pipeline works.

In [ ]:
!python scripts/build_passages.py

In [ ]:
!python scripts/train.py

In [ ]:
!python scripts/evaluate.py

## 5. Automatic tests (should say `13 passed`)

In [ ]:
!python -m pytest -q

## 6. Look at the results

In [ ]:
import json
from IPython.display import Image, display
r = json.load(open('static/reports/metrics.json'))
for k, m in r['models'].items():
    print(f"{m['name']:<20} accuracy={m['accuracy']}  macro-F1={m['macro_f1']}")
for k, m in r['search']['methods'].items():
    print(f"{m['name']:<28} top-1={m['top1']}  top-3={m['top3']}")
print("off-topic -> Not found:", r['offtopic']['rejected'], "/", r['offtopic']['total'])
display(Image('static/reports/confusion_matrix_logreg.png'))
display(Image('static/reports/threshold.png'))
pd.DataFrame(r['robustness'])

## 7. Try the models by hand
Change the questions and run again.

In [ ]:
import sys; sys.path.insert(0, '.')
from core.search import SearchEngine
from core.classifier import TopicClassifier
from core.topics import topic_names
eng, clf, names = SearchEngine.load(), TopicClassifier.load(), topic_names()

def ask(q):
    topic, conf = clf.predict(q)
    found, hits = eng.search(q)
    print(f"\nQ: {q}")
    if not found:
        print("   -> Not found in the course"); return
    print(f"   topic: {names[topic]} ({conf:.0%})")
    for h in hits:
        print(f"   Lecture {h['lecture']} · Slide {h['slide']}  ({h['score']}): {h['text'][:110]}...")

for q in ["What is an admissible heuristic?", "difference between bfs and dfs",
          "when does alpha beta prune", "ايه هو البحث بالعرض", "who won the world cup"]:
    ask(q)

## 8. Run the website inside Colab
Run the cell, then click the link it prints (it opens the app in a new tab). Student code: anything, e.g. `S001`. Lecturer password: `lecturer123`.
To stop the server: Runtime → Interrupt / restart.

In [ ]:
import threading, time, sys
sys.path.insert(0, '/content/studymate')
from app import app
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(3)
from google.colab.output import eval_js
print("Open StudyMate:", eval_js("google.colab.kernel.proxyPort(5000)"))

In [ ]:
# Optional: show the app inside the notebook instead of a new tab
from google.colab import output
output.serve_kernel_port_as_iframe(5000, height=800)

## 9. Check the Excel log after using the website

In [ ]:
for sheet in ['Questions', 'Answers', 'Visits', 'Feedback']:
    print(sheet); display(pd.read_excel('logs.xlsx', sheet_name=sheet).tail(5))

## 10. (Optional) download the updated logs / reports

In [ ]:
from google.colab import files
files.download('logs.xlsx')